# Heart Sound Analysis -- Automated Benchmark & Audit Pipeline
### Leakage-Aware Evaluation, 5 Architectural Ablations, Explanation Audits & Paper Deliverables

**Instructions:**
1. Go to **Runtime -> Change runtime type** and ensure **T4 GPU** is selected.
2. Click **Runtime -> Run all** (or press `Ctrl + F9`).
3. The notebook will automatically execute the entire 24-experiment benchmark, audit explanation faithfulness, compile all LaTeX tables and macros, and download the results.


## 0. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability (CUDA or Apple Silicon MPS)
import sys
import shutil
import torch

print(f'PyTorch Version: {torch.__version__}')
cuda_avail = torch.cuda.is_available()
mps_avail = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
print(f'CUDA Available: {cuda_avail}')
print(f'Apple Silicon MPS Available: {mps_avail}')

if cuda_avail:
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
    if shutil.which('nvidia-smi'):
        !nvidia-smi
elif mps_avail:
    print('Using Apple Silicon Metal Performance Shaders (MPS) GPU acceleration.')
else:
    print('Running on CPU.')


In [ ]:
# Install required Python dependencies
%pip install -q librosa scipy numpy pandas torch torchvision tqdm shap h5py pytest

## 1. Repository Setup & Drive Mount (Optional)

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# 1. Mount Google Drive if in Colab
is_colab = os.path.exists('/content')
if is_colab:
    if not Path('/content/drive/MyDrive').exists():
        from google.colab import drive
        drive.mount('/content/drive')
    print('Google Drive mounted.')
    
    # 2. Always use fresh GitHub code in /content/heart-sound-analysis
    repo_dir = Path('/content/heart-sound-analysis')
    if not repo_dir.exists():
        print('Cloning latest codebase from GitHub...')
        !git clone https://github.com/urvagandhi/heart-sound-analysis.git /content/heart-sound-analysis
    else:
        print('Pulling latest updates from GitHub...')
        !git -C /content/heart-sound-analysis pull
        
    # 3. Locate dataset on Google Drive
    drive_candidates = [
        Path('/content/drive/MyDrive/heart-sound-analysis'),
        Path('/content/drive/MyDrive/RMS'),
        Path('/content/drive/MyDrive/CSE/College/RMS'),
    ]
    drive_candidates.extend(Path.home().glob('Library/CloudStorage/GoogleDrive-*/My Drive/heart-sound-analysis'))
    drive_rms = next((d for d in drive_candidates if d.exists()), None)
    
    if drive_rms and (drive_rms / 'physionet_2016').exists():
        drive_data = drive_rms / 'physionet_2016'
        target_data = repo_dir / 'physionet_2016'
        
        # Link subdirectories (spectrograms, raw) from Drive
        for subfolder in ['spectrograms', 'raw']:
            src_sub = drive_data / subfolder
            dst_sub = target_data / subfolder
            if src_sub.exists():
                if dst_sub.is_symlink():
                    dst_sub.unlink()
                if not dst_sub.exists():
                    print(f'Linking {subfolder}: {src_sub} -> {dst_sub}')
                    os.symlink(str(src_sub), str(dst_sub))
                else:
                    print(f'{subfolder} already accessible at {dst_sub}')
                    
        # Also link physionet_2016.zip if present in Drive
        zip_drive = drive_rms / 'physionet_2016.zip'
        zip_target = repo_dir / 'physionet_2016.zip'
        if zip_drive.exists() and not zip_target.exists():
            print(f'Linking zip file: {zip_drive} -> {zip_target}')
            os.symlink(str(zip_drive), str(zip_target))
            
        # 4. Link results and output directories to Drive
        SAVE_TO_SHARED = True
        
        target_results = repo_dir / 'results'
        drive_results = None
        if SAVE_TO_SHARED:
            try:
                candidate_results = drive_rms / 'results'
                candidate_results.mkdir(parents=True, exist_ok=True)
                test_f = candidate_results / '.perm_test'
                test_f.touch()
                test_f.unlink()
                drive_results = candidate_results
                print(f'Saving results to Drive folder: {drive_results}')
            except (PermissionError, OSError):
                print('Notice: Shared folder is not writable. Falling back to personal My Drive.')
        
        if drive_results is None:
            drive_results = Path('/content/drive/MyDrive/heart_sound_results')
            drive_results.mkdir(parents=True, exist_ok=True)
            print(f'Saving results to your personal My Drive: {drive_results}')
            
        if target_results.is_symlink():
            print(f'Results symlink already active: {target_results} -> {target_results.resolve()}')
        else:
            if target_results.exists():
                for item in target_results.iterdir():
                    dst = drive_results / item.name
                    if not dst.exists():
                        if item.is_file(): shutil.copy2(item, dst)
                        elif item.is_dir(): shutil.copytree(item, dst)
                shutil.rmtree(target_results)
            os.symlink(str(drive_results), str(target_results))
            print(f'Linked results directory: {drive_results} -> {target_results}')

        # Also link Review 3/output checkpoints
        target_out = repo_dir / 'Review 3' / 'output'
        drive_out = drive_rms / 'Review 3' / 'output'
        if drive_out.exists():
            if target_out.is_symlink():
                pass
            else:
                if target_out.exists():
                    shutil.rmtree(target_out)
                os.symlink(str(drive_out), str(target_out))
                print(f'Linked Review 3/output directory: {drive_out} -> {target_out}')
    else:
        print('Warning: physionet_2016 not found in Drive heart-sound-analysis or RMS folder. Using repo files.')
else:
    print('Running in local environment (MacBook).')
    repo_dir = Path.cwd()
    target_data = repo_dir / 'physionet_2016'
    for subfolder in ['spectrograms', 'raw']:
        dst_sub = target_data / subfolder
        if not dst_sub.exists() or (subfolder == 'spectrograms' and len(list(dst_sub.glob('*.npy'))) == 0):
            for mac_d in Path.home().glob('Library/CloudStorage/GoogleDrive-*/My Drive/heart-sound-analysis'):
                src_sub = mac_d / 'physionet_2016' / subfolder
                if src_sub.exists() and len(list(src_sub.iterdir())) > 0:
                    if dst_sub.is_symlink():
                        dst_sub.unlink()
                    if not dst_sub.exists():
                        print(f'Linking {subfolder} from Google Drive: {src_sub} -> {dst_sub}')
                        os.symlink(str(src_sub), str(dst_sub))
                    break

os.chdir(str(repo_dir))
print(f'Active working directory: {Path.cwd()}')

## 2. Dataset Verification & Extraction

In [ ]:
import zipfile
data_dir = Path("physionet_2016")
raw_dir = data_dir / "raw"
spec_dir = data_dir / "spectrograms"
zip_file = Path("physionet_2016.zip")

# Fallback linking: if spec_dir missing or empty, link from Google Drive directly
if not spec_dir.exists() or len(list(spec_dir.glob("*.npy"))) == 0:
    candidates = [
        Path("/content/drive/MyDrive/heart-sound-analysis/physionet_2016/spectrograms"),
        Path("/content/drive/MyDrive/RMS/physionet_2016/spectrograms"),
        Path("/content/drive/MyDrive/CSE/College/RMS/physionet_2016/spectrograms"),
    ]
    candidates.extend(Path.home().glob("Library/CloudStorage/GoogleDrive-*/My Drive/heart-sound-analysis/physionet_2016/spectrograms"))
    for cand in candidates:
        if cand.exists() and len(list(cand.glob("*.npy"))) > 0:
            if spec_dir.is_symlink():
                spec_dir.unlink()
            elif spec_dir.is_dir() and not any(spec_dir.iterdir()):
                spec_dir.rmdir()
            if not spec_dir.exists():
                print(f"Linking spectrograms from Drive: {cand} -> {spec_dir}")
                os.symlink(str(cand), str(spec_dir))
                break

# Extract if zip exists and spectrograms not yet unpacked
if zip_file.exists() and (not spec_dir.exists() or len(list(spec_dir.glob("*.npy"))) < 100):
    print("Extracting physionet_2016.zip into physionet_2016/...")
    with zipfile.ZipFile(zip_file, "r") as z:
        for m in z.infolist():
            rel_name = m.filename.split("physionet_2016/")[-1]
            if not rel_name:
                continue
            target = data_dir / rel_name
            if m.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, "wb") as dst:
                    dst.write(src.read())
    print("Extraction complete!")

print(f"Spectrograms available: {len(list(spec_dir.glob('*.npy')))} / 3541")
print(f"Metadata CSV exists: {(data_dir / 'metadata.csv').exists()}")

## 3. Generate Splits & Run Duplicate Leakage Audit

In [ ]:
# Generate deduplicated (3 seeds) and LODO splits
import sys
py_exec = sys.executable

!{py_exec} 'Review 3/make_splits.py' --mode dedup --seeds 42 43 44
!{py_exec} 'Review 3/make_splits.py' --mode lodo --holdouts a b e f --seed 42

# Check twin duplicate performance vs unique test accuracy
!{py_exec} 'Review 3/verify_duplicates.py'


## 4. Run All 24 Benchmark Experiments Sequentially

In [ ]:
# Dry-run verification (24 planned runs)
import sys
py_exec = sys.executable
!{py_exec} 'Review 3/run_all.py' --dry-run

# Execute full 24-run benchmark (resume-safe: skips already completed runs)
!{py_exec} 'Review 3/run_all.py'


## 5. Run Population-Level Explanation Audit Suite

In [ ]:
# Execute audit on primary model: dedup_se_mha_seed42
import sys
py_exec = sys.executable
!{py_exec} 'Review 3/audit/run_audit.py' --tag dedup_se_mha_seed42


## 6. Aggregate Deliverables & Compile LaTeX Paper

In [ ]:
# Aggregate all JSON artifacts into LaTeX tables, macros, and findings
import sys, shutil
py_exec = sys.executable
!{py_exec} 'Review 3/aggregate_results.py'

# Run unit tests to verify consistency
!{py_exec} -m pytest -q

# Compile PDF using pdflatex if available
if shutil.which('pdflatex'):
    %cd 'Paper Writing'
    !pdflatex -interaction=nonstopmode paper.tex
    !bibtex paper
    !pdflatex -interaction=nonstopmode paper.tex
    !pdflatex -interaction=nonstopmode paper.tex
    %cd ..
elif shutil.which('apt-get'):
    !apt-get update -qq && apt-get install -y -qq texlive-latex-base texlive-fonts-recommended texlive-fonts-extra texlive-latex-extra
    %cd 'Paper Writing'
    !pdflatex -interaction=nonstopmode paper.tex
    !bibtex paper
    !pdflatex -interaction=nonstopmode paper.tex
    !pdflatex -interaction=nonstopmode paper.tex
    %cd ..
else:
    print('pdflatex not found on system. Skipping PDF compilation.')


## 7. Download Results Archive

In [ ]:
from pathlib import Path

# Create zip of all experimental results and the compiled paper PDF
output_archive = 'benchmark_results_and_paper.zip'
!zip -q -r benchmark_results_and_paper.zip results 'Paper Writing/generated' 
print(f'Created archive: {output_archive}')

try:
    from google.colab import files
    files.download(output_archive)
except Exception:
    print(f'Saved locally at: {Path.cwd() / output_archive}')
